<br>

# FDBS

_Script_ para obter dados usando LXML.

Quero simplificar!

<br>

Michel Metran\
Data: 02.11.2025\
Atualizado em: 21.09.2026


In [ ]:
from pathlib import Path

import pyFDBS
from pyFDBS.requests.logger import FBDSLogger
from pyFDBS.requests.web import FBDS

In [ ]:
# import fbds

<br>

---

## Pastas


In [ ]:
project_path = Path('.').absolute().parents[2]
print(project_path)

# Diretório de saída
data_path = project_path / 'data'

logs_path = data_path / 'log'
logs_path.mkdir(parents=True, exist_ok=True)

output_path = data_path / 'output'
output_path.mkdir(parents=True, exist_ok=True)
output_path

<br>

---

## FBDS

Instancia a classe


In [ ]:
fdbs = FBDS(temp_path=output_path)

<br>

---

### States

Lista estados


In [ ]:
states = fdbs.get_states()
states

Obtem um estado específico


In [ ]:
state = fdbs.get_state(uf='SP')
state

<br>

---

### Municipios

Lista municípios de um estado.


In [ ]:
municipios = fdbs.get_municipalities(uf="SP")
municipios

<br>

---

### Layers

Lista layers de um município.


In [ ]:
# Inicializa o logger uma única vez antes do loop
logger = FBDSLogger(log_dir=logs_path)
logger.start_download_session()

for muninicio in municipios:
    municipio_name = muninicio["name"]
    print(municipio_name)

    # Listas Disponíveis
    lyrs = fdbs.get_layers(
        municipality=municipio_name,
        uf="SP",
    )
    for lyr in lyrs:
        layer_name = lyr["name"]
        print(layer_name)

        # Acessa o layer
        lyr = fdbs.get_layer(
            municipality=municipio_name,
            uf="SP",
            layer=layer_name,
        )

        # Lista de arquivos para download (do exemplo)
        files_to_download = fdbs.get_links(
            url=lyr["url"],
            ignore_first=2,
        )
        # print(files_to_download)

        # Download usando threads (melhor para downloads)
        results_thread = pyFDBS.download_files_parallel(
            url_list=files_to_download,
            output_dir=output_path,
            # Número de downloads simultâneos
            max_concurrent=4,
        )

# Finaliza a sessão após todo o processamento
logger.end_download_session()